# 19_patchtst_sequence_subset.ipynb

Ελεγχόμενο notebook για sequence-aligned πείραμα σε σταθερό four-park subset.

Το notebook χρησιμοποιεί αποκλειστικά τα canonical split artifacts:

- `data/processed/train_final.csv`
- `data/processed/val_final.csv`
- `data/processed/test_final.csv`

Τα δύο μοντέλα είναι:

- Flattened-window XGBoost πάνω σε sequence windows `[n_windows, 24, 41]`.
- Pure PyTorch PatchTST-Lite Transformer με input `[batch, 24, 41]` και output `[batch, 1]`.

Τα προεπιλεγμένα flags είναι ασφαλή: δεν γίνεται training, δεν γίνεται test evaluation και δεν γράφονται exports.

## Συμβόλαιο Στόχου Ακολουθίας

Για κάθε split, με ανεξάρτητη επεξεργασία ανά `park_id`:

1. Οι γραμμές ταξινομούνται κατά `park_id` και `timestamp`.
2. Η χρονοσειρά σπάει σε συνεχόμενα segments με βάση timestamp gaps.
3. Το X περιέχει τα προηγούμενα `24` timesteps.
4. Το y είναι το αμέσως επόμενο `Power_Output_Normalized`.
5. Δεν δημιουργούνται windows που περνούν park boundaries, split boundaries ή timestamp gaps.

Το αναμενόμενο τελικό feature tensor είναι `[n_windows, 24, 41]`.

In [ ]:
# ============================================================
# NB19 | Imports, αναπαραγωγιμότητα και σταθερές
# ============================================================

from __future__ import annotations

import copy
import itertools
import random
import time
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset


try:
    display
except NameError:
    def display(obj: Any) -> None:
        print(obj)


SEED = 42
EXPECTED_SELECTED_PARKS = ['00183', '00198', '00303', '00427']
SELECTED_PARKS = EXPECTED_SELECTED_PARKS.copy()

TARGET_COLUMN = 'Power_Output_Normalized'
PARK_ID_COLUMN = 'park_id'
TIMESTAMP_COLUMN = 'timestamp'
TEST_FLAG_COLUMN = 'test_flag'
BASELINE_COLUMN = 'Baseline_Prediction'
TURBINE_COLUMN = 'turbine'

EXCLUDED_COLUMNS = {
    TARGET_COLUMN,
    PARK_ID_COLUMN,
    TIMESTAMP_COLUMN,
    TEST_FLAG_COLUMN,
    BASELINE_COLUMN,
    TURBINE_COLUMN,
}

LOOKBACK_STEPS = 24
EXPECTED_FEATURE_COUNT = 41
EXPECTED_FLATTENED_FEATURE_COUNT = LOOKBACK_STEPS * EXPECTED_FEATURE_COUNT
EXPECTED_FREQ = pd.Timedelta(hours=1)

PATCH_LEN = 4
PATCH_STRIDE = 2
D_MODEL = 64
N_HEADS = 4
N_LAYERS = 2
DROPOUT = 0.1
BATCH_SIZE = 512
LEARNING_RATE = 0.001
WEIGHT_DECAY = 1e-5
EPOCHS_REQUESTED = 30
EARLY_STOPPING_PATIENCE = 5
PATCHTST_MODEL_ID = 'patchtst_lite_p4_s2_d64_h4_l2_do0_1'

XGBOOST_GRID = {
    'n_estimators': [300, 500],
    'max_depth': [4, 6],
    'learning_rate': [0.03, 0.05],
    'subsample': [0.8],
    'colsample_bytree': [0.8],
}

SMOKE_MODE = True
RUN_SMOKE_TRAINING = False
RUN_TEST_EVALUATION = False
EXPORT_RESULTS = False

SMOKE_EPOCHS = 2
SMOKE_PATIENCE = 1
SMOKE_MAX_WINDOWS = {
    'train': 512,
    'validation': 256,
    'test': 256,
}

VALIDATION_RANKING_COLUMNS = ['MAE', 'RMSE', 'R2']
VALIDATION_RANKING_ASCENDING = [True, True, False]


def set_reproducibility(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if hasattr(torch.backends, 'cudnn'):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    y_true = np.asarray(y_true, dtype=np.float64).reshape(-1)
    y_pred = np.asarray(y_pred, dtype=np.float64).reshape(-1)
    if y_true.shape != y_pred.shape:
        raise ValueError(f'Ασύμβατα metric shapes: y_true={y_true.shape}, y_pred={y_pred.shape}')
    residual = y_true - y_pred
    mae = float(np.mean(np.abs(residual)))
    rmse = float(np.sqrt(np.mean(residual**2)))
    ss_res = float(np.sum(residual**2))
    ss_tot = float(np.sum((y_true - np.mean(y_true)) ** 2))
    r2 = float(1.0 - ss_res / ss_tot) if ss_tot > 0 else np.nan
    return {'MAE': mae, 'RMSE': rmse, 'R2': r2}


def rank_validation_metrics(metrics_df: pd.DataFrame) -> pd.DataFrame:
    if metrics_df.empty:
        return metrics_df
    ranked = metrics_df.sort_values(
        VALIDATION_RANKING_COLUMNS,
        ascending=VALIDATION_RANKING_ASCENDING,
        kind='mergesort',
    ).reset_index(drop=True)
    ranked.insert(0, 'validation_rank', np.arange(1, len(ranked) + 1))
    return ranked


set_reproducibility(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('SEED:', SEED)
print('DEVICE:', DEVICE)
print('SMOKE_MODE:', SMOKE_MODE)
print('RUN_SMOKE_TRAINING:', RUN_SMOKE_TRAINING)
print('RUN_TEST_EVALUATION:', RUN_TEST_EVALUATION)
print('EXPORT_RESULTS:', EXPORT_RESULTS)

In [ ]:
# ============================================================
# NB19 | Paths και προαιρετικές σταθερές εξόδων
# ============================================================

def find_project_root(start_path: Path) -> Path:
    current = start_path.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'data').exists() and (candidate / 'notebooks').exists():
            return candidate
    raise FileNotFoundError('Δεν βρέθηκε project root με φακέλους data/ και notebooks/.')


def get_file_state(path: Path) -> dict[str, Any]:
    if not path.exists():
        return {'exists': False, 'size': None, 'mtime_ns': None}
    stat = path.stat()
    return {'exists': True, 'size': stat.st_size, 'mtime_ns': stat.st_mtime_ns}


PROJECT_ROOT = find_project_root(Path.cwd())
DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'

TRAIN_PATH = DATA_PROCESSED / 'train_final.csv'
VAL_PATH = DATA_PROCESSED / 'val_final.csv'
TEST_PATH = DATA_PROCESSED / 'test_final.csv'
BASELINE_METRICS_PATH = DATA_PROCESSED / 'baseline_metrics.csv'
BASELINE_METRICS_INITIAL_STATE = get_file_state(BASELINE_METRICS_PATH)

OUTPUT_DIR = DATA_PROCESSED / 'diagnostics' / 'nn_sequence_subset_patchtst'
RUN_MANIFEST_PATH = OUTPUT_DIR / 'nn_sequence_subset_patchtst_run_manifest.csv'
WINDOW_AUDIT_PATH = OUTPUT_DIR / 'nn_sequence_subset_patchtst_window_audit.csv'
XGBOOST_VALIDATION_METRICS_PATH = OUTPUT_DIR / 'nn_sequence_subset_patchtst_xgboost_validation_metrics.csv'
XGBOOST_SELECTED_TEST_METRICS_PATH = OUTPUT_DIR / 'nn_sequence_subset_patchtst_xgboost_selected_test_metrics.csv'
TRAINING_HISTORY_PATH = OUTPUT_DIR / 'nn_sequence_subset_patchtst_training_history.csv'
VALIDATION_METRICS_PATH = OUTPUT_DIR / 'nn_sequence_subset_patchtst_validation_metrics.csv'
SELECTED_TEST_METRICS_PATH = OUTPUT_DIR / 'nn_sequence_subset_patchtst_selected_test_metrics.csv'

for label, path in [('train', TRAIN_PATH), ('validation', VAL_PATH), ('test', TEST_PATH)]:
    if not path.exists():
        raise FileNotFoundError(f'Λείπει το απαιτούμενο {label} split: {path}')

print('PROJECT_ROOT:', PROJECT_ROOT)
print('TRAIN_PATH:', TRAIN_PATH)
print('VAL_PATH:', VAL_PATH)
print('TEST_PATH:', TEST_PATH)
print('Προαιρετικός φάκελος εξόδων:', OUTPUT_DIR)
print('Το baseline metrics path είναι μόνο για ανάγνωση στο NB19:', BASELINE_METRICS_PATH)

## Φόρτωση Και Έλεγχος Canonical Splits

Ο loader διαβάζει μόνο τις γραμμές των selected parks από τα canonical split artifacts. Το `park_id` διατηρείται ως zero-padded string και το `timestamp` γίνεται parse πριν από οποιαδήποτε κατασκευή ακολουθιών.

In [ ]:
# ============================================================
# NB19 | Φόρτωση επιλεγμένων parks από τα canonical split artifacts
# ============================================================

REQUIRED_COLUMNS = {
    PARK_ID_COLUMN,
    TIMESTAMP_COLUMN,
    TEST_FLAG_COLUMN,
    TARGET_COLUMN,
    BASELINE_COLUMN,
}


def normalize_park_id(series: pd.Series) -> pd.Series:
    return series.astype(str).str.replace('.0', '', regex=False).str.zfill(5)


def read_filtered_split(path: Path, split_name: str, selected_parks: list[str]) -> pd.DataFrame:
    header = pd.read_csv(path, nrows=0).columns
    missing = sorted(REQUIRED_COLUMNS - set(header))
    if missing:
        raise KeyError(f'Λείπουν required columns από το {split_name}: {missing}')

    selected = set(selected_parks)
    frames: list[pd.DataFrame] = []
    for chunk in pd.read_csv(path, dtype={PARK_ID_COLUMN: 'string'}, chunksize=100_000):
        chunk[PARK_ID_COLUMN] = normalize_park_id(chunk[PARK_ID_COLUMN])
        filtered = chunk.loc[chunk[PARK_ID_COLUMN].isin(selected)].copy()
        if not filtered.empty:
            frames.append(filtered)

    if not frames:
        raise ValueError(f'Δεν βρέθηκαν γραμμές στο {split_name} για τα selected parks: {selected_parks}')

    df = pd.concat(frames, ignore_index=True)
    df[TIMESTAMP_COLUMN] = pd.to_datetime(df[TIMESTAMP_COLUMN], errors='raise')
    df[PARK_ID_COLUMN] = normalize_park_id(df[PARK_ID_COLUMN])
    return df


train_raw = read_filtered_split(TRAIN_PATH, 'train', SELECTED_PARKS)
val_raw = read_filtered_split(VAL_PATH, 'validation', SELECTED_PARKS)
test_raw = read_filtered_split(TEST_PATH, 'test', SELECTED_PARKS)

split_frames = {
    'train': train_raw,
    'validation': val_raw,
    'test': test_raw,
}

for split_name, df in split_frames.items():
    observed_parks = sorted(df[PARK_ID_COLUMN].unique().tolist())
    if observed_parks != SELECTED_PARKS:
        raise ValueError(f'Μη αναμενόμενα parks στο {split_name}: {observed_parks}')
    if df.duplicated(subset=[PARK_ID_COLUMN, TIMESTAMP_COLUMN]).any():
        raise ValueError(f'Βρέθηκαν duplicate (park_id, timestamp) rows στο {split_name}.')
    core_nulls = int(df[[PARK_ID_COLUMN, TIMESTAMP_COLUMN, TARGET_COLUMN]].isnull().sum().sum())
    if core_nulls:
        raise ValueError(f'Βρέθηκαν null values στις core {split_name} columns: {core_nulls}')

assert set(train_raw[TEST_FLAG_COLUMN].dropna().astype(int).unique()) <= {0}
assert set(val_raw[TEST_FLAG_COLUMN].dropna().astype(int).unique()) <= {0}
assert set(test_raw[TEST_FLAG_COLUMN].dropna().astype(int).unique()) <= {1}

split_summary_df = pd.DataFrame(
    [
        {
            'split': split_name,
            'rows': len(df),
            'parks': df[PARK_ID_COLUMN].nunique(),
            'min_timestamp': df[TIMESTAMP_COLUMN].min(),
            'max_timestamp': df[TIMESTAMP_COLUMN].max(),
        }
        for split_name, df in split_frames.items()
    ]
)

display(split_summary_df)

## Συμβόλαιο Features Και Train-Only Scaling

Η επιλογή learned features προκύπτει μόνο από το train subset. Τα validation και test χρησιμοποιούνται μόνο για έλεγχο διαθεσιμότητας των train-selected columns και για εφαρμογή του scaler που έχει γίνει fit στο train.

Ο στόχος παραμένει unscaled και το `turbine` δεν κωδικοποιείται.

In [ ]:
# ============================================================
# NB19 | Train-inferred numeric features και scaling
# ============================================================

FEATURE_SELECTION_SOURCE = 'train'
SCALER_FIT_SOURCE = 'train'

candidate_feature_cols = [col for col in train_raw.columns if col not in EXCLUDED_COLUMNS]
numeric_feature_cols = [
    col for col in candidate_feature_cols
    if pd.api.types.is_numeric_dtype(train_raw[col])
]
non_numeric_excluded = sorted(set(candidate_feature_cols) - set(numeric_feature_cols))

if not numeric_feature_cols:
    raise ValueError('Δεν βρέθηκαν train-inferred numeric learned feature columns.')

for blocked in EXCLUDED_COLUMNS:
    if blocked in numeric_feature_cols:
        raise ValueError(f'Blocked column μπήκε στα features: {blocked}')

if len(numeric_feature_cols) != EXPECTED_FEATURE_COUNT:
    raise ValueError(
        f'Αναμενόταν feature_count={EXPECTED_FEATURE_COUNT}, '
        f'αλλά το train-inferred contract έδωσε {len(numeric_feature_cols)}.'
    )

for split_name, df in split_frames.items():
    missing = [col for col in numeric_feature_cols if col not in df.columns]
    if missing:
        raise KeyError(f'Λείπουν train-selected features από το {split_name}: {missing}')
    selected_nulls = int(df[[TARGET_COLUMN, *numeric_feature_cols]].isnull().sum().sum())
    if selected_nulls:
        raise ValueError(f'Null target/feature values στο {split_name}: {selected_nulls}')
    selected_values = df[[TARGET_COLUMN, *numeric_feature_cols]].to_numpy(dtype=np.float64)
    if not np.isfinite(selected_values).all():
        raise ValueError(f'Βρέθηκαν non-finite target/feature values στο {split_name}.')

scaler = StandardScaler()
train_scaled_values = scaler.fit_transform(train_raw[numeric_feature_cols]).astype(np.float32)
val_scaled_values = scaler.transform(val_raw[numeric_feature_cols]).astype(np.float32)
test_scaled_values = scaler.transform(test_raw[numeric_feature_cols]).astype(np.float32)


def make_scaled_sequence_frame(df: pd.DataFrame, scaled_values: np.ndarray) -> pd.DataFrame:
    out = df[[PARK_ID_COLUMN, TIMESTAMP_COLUMN, TARGET_COLUMN]].copy()
    out[numeric_feature_cols] = scaled_values
    return out


train_scaled_df = make_scaled_sequence_frame(train_raw, train_scaled_values)
val_scaled_df = make_scaled_sequence_frame(val_raw, val_scaled_values)
test_scaled_df = make_scaled_sequence_frame(test_raw, test_scaled_values)

feature_audit_df = pd.DataFrame(
    {
        'feature': numeric_feature_cols,
        'train_dtype': [str(train_raw[col].dtype) for col in numeric_feature_cols],
    }
)

print('Πλήθος numeric learned features:', len(numeric_feature_cols))
print('Non-numeric candidate columns που εξαιρέθηκαν:', non_numeric_excluded or 'none')
print('Πολιτική target scaling: ο στόχος παραμένει unscaled.')
display(feature_audit_df.head(20))

## Gap-Safe Κατασκευή Ακολουθιών

Για `LOOKBACK_STEPS = 24`, κάθε feature window χρησιμοποιεί τις γραμμές `[i, ..., i+23]`, ενώ ο στόχος είναι η γραμμή `i+24`. Η κατασκευή γίνεται ξεχωριστά για train, validation και test, άρα δεν περνάει split boundary.

In [ ]:
# ============================================================
# NB19 | Leakage-safe sliding windows μόνο μέσα σε κάθε split
# ============================================================

def build_sequence_split(
    df: pd.DataFrame,
    split_name: str,
    feature_cols: list[str],
    lookback_steps: int,
    expected_freq: pd.Timedelta,
) -> tuple[np.ndarray, np.ndarray, pd.DataFrame, pd.DataFrame]:
    X_parts: list[np.ndarray] = []
    y_parts: list[float] = []
    meta_rows: list[dict[str, Any]] = []
    audit_rows: list[dict[str, Any]] = []

    sorted_df = df.sort_values([PARK_ID_COLUMN, TIMESTAMP_COLUMN], kind='mergesort').reset_index(drop=True)

    for park_id, park_df in sorted_df.groupby(PARK_ID_COLUMN, sort=False):
        park_df = park_df.sort_values(TIMESTAMP_COLUMN, kind='mergesort').reset_index(drop=True)
        gap_start = park_df[TIMESTAMP_COLUMN].diff().ne(expected_freq)
        segment_ids = gap_start.cumsum()

        for segment_id, segment_df in park_df.groupby(segment_ids, sort=False):
            segment_df = segment_df.reset_index(drop=True)
            segment_len = len(segment_df)
            n_windows = max(segment_len - lookback_steps, 0)

            audit_rows.append(
                {
                    'split': split_name,
                    'park_id': park_id,
                    'segment_id': int(segment_id),
                    'segment_start_timestamp': segment_df[TIMESTAMP_COLUMN].iloc[0],
                    'segment_end_timestamp': segment_df[TIMESTAMP_COLUMN].iloc[-1],
                    'segment_rows': segment_len,
                    'lookback_steps': lookback_steps,
                    'windows': n_windows,
                }
            )

            if n_windows == 0:
                continue

            feature_values = segment_df[feature_cols].to_numpy(dtype=np.float32)
            target_values = segment_df[TARGET_COLUMN].to_numpy(dtype=np.float32)
            timestamps = segment_df[TIMESTAMP_COLUMN].reset_index(drop=True)

            for start_idx in range(n_windows):
                feature_start_idx = start_idx
                feature_end_exclusive = start_idx + lookback_steps
                target_idx = start_idx + lookback_steps

                window_start_ts = timestamps.iloc[feature_start_idx]
                window_end_ts = timestamps.iloc[feature_end_exclusive - 1]
                target_ts = timestamps.iloc[target_idx]

                if target_ts - window_end_ts != expected_freq:
                    raise ValueError(
                        f'Το target timestamp δεν είναι η επόμενη γραμμή μετά το feature window στο {split_name}, '
                        f'park={park_id}, segment={segment_id}.'
                    )

                X_parts.append(feature_values[feature_start_idx:feature_end_exclusive])
                y_parts.append(float(target_values[target_idx]))
                meta_rows.append(
                    {
                        'split': split_name,
                        'park_id': park_id,
                        'segment_id': int(segment_id),
                        'window_start_timestamp': window_start_ts,
                        'window_end_timestamp': window_end_ts,
                        'target_timestamp': target_ts,
                    }
                )

    if X_parts:
        X = np.stack(X_parts).astype(np.float32)
        y = np.asarray(y_parts, dtype=np.float32)
    else:
        X = np.empty((0, lookback_steps, len(feature_cols)), dtype=np.float32)
        y = np.empty((0,), dtype=np.float32)

    meta_df = pd.DataFrame(meta_rows)
    audit_df = pd.DataFrame(audit_rows)
    return X, y, meta_df, audit_df


X_train_seq, y_train_seq, train_window_meta_df, train_window_audit_df = build_sequence_split(
    train_scaled_df, 'train', numeric_feature_cols, LOOKBACK_STEPS, EXPECTED_FREQ
)
X_val_seq, y_val_seq, val_window_meta_df, val_window_audit_df = build_sequence_split(
    val_scaled_df, 'validation', numeric_feature_cols, LOOKBACK_STEPS, EXPECTED_FREQ
)
X_test_seq, y_test_seq, test_window_meta_df, test_window_audit_df = build_sequence_split(
    test_scaled_df, 'test', numeric_feature_cols, LOOKBACK_STEPS, EXPECTED_FREQ
)

window_audit_df = pd.concat(
    [train_window_audit_df, val_window_audit_df, test_window_audit_df],
    ignore_index=True,
)

sequence_summary_df = pd.DataFrame(
    [
        {'split': 'train', 'windows': len(X_train_seq), 'shape': X_train_seq.shape},
        {'split': 'validation', 'windows': len(X_val_seq), 'shape': X_val_seq.shape},
        {'split': 'test', 'windows': len(X_test_seq), 'shape': X_test_seq.shape},
    ]
)

display(sequence_summary_df)
display(window_audit_df.head(12))

In [ ]:
# ============================================================
# NB19 | Smoke caps και dataloaders
# ============================================================

def cap_sequence_arrays(
    X: np.ndarray,
    y: np.ndarray,
    meta_df: pd.DataFrame,
    split_name: str,
    smoke_mode: bool,
) -> tuple[np.ndarray, np.ndarray, pd.DataFrame]:
    if not smoke_mode:
        return X, y, meta_df

    cap = SMOKE_MAX_WINDOWS[split_name]
    capped_n = min(cap, len(X))
    return X[:capped_n], y[:capped_n], meta_df.iloc[:capped_n].reset_index(drop=True)


X_train_model, y_train_model, train_model_meta_df = cap_sequence_arrays(
    X_train_seq, y_train_seq, train_window_meta_df, 'train', SMOKE_MODE
)
X_val_model, y_val_model, val_model_meta_df = cap_sequence_arrays(
    X_val_seq, y_val_seq, val_window_meta_df, 'validation', SMOKE_MODE
)
X_test_model, y_test_model, test_model_meta_df = cap_sequence_arrays(
    X_test_seq, y_test_seq, test_window_meta_df, 'test', SMOKE_MODE
)


def make_loader(X: np.ndarray, y: np.ndarray, batch_size: int, shuffle: bool, seed: int) -> DataLoader:
    dataset = TensorDataset(
        torch.from_numpy(X.astype(np.float32)),
        torch.from_numpy(y.astype(np.float32).reshape(-1, 1)),
    )
    generator = None
    if shuffle:
        generator = torch.Generator()
        generator.manual_seed(seed)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, generator=generator)


if min(len(X_train_model), len(X_val_model), len(X_test_model)) == 0:
    raise ValueError('At least one split has no sequence windows after smoke/full selection.')

train_loader = make_loader(X_train_model, y_train_model, BATCH_SIZE, shuffle=True, seed=SEED)
val_loader = make_loader(X_val_model, y_val_model, BATCH_SIZE, shuffle=False, seed=SEED)
test_loader = make_loader(X_test_model, y_test_model, BATCH_SIZE, shuffle=False, seed=SEED)

first_batch_X, first_batch_y = next(iter(train_loader))
first_batch_shape = tuple(first_batch_X.shape)
if first_batch_shape[1:] != (LOOKBACK_STEPS, len(numeric_feature_cols)):
    raise ValueError(f'Μη αναμενόμενο πρώτο batch shape: {first_batch_shape}')

model_window_summary_df = pd.DataFrame(
    [
        {'split': 'train', 'model_windows': len(X_train_model), 'source_windows': len(X_train_seq)},
        {'split': 'validation', 'model_windows': len(X_val_model), 'source_windows': len(X_val_seq)},
        {'split': 'test', 'model_windows': len(X_test_model), 'source_windows': len(X_test_seq)},
    ]
)

print('First batch X shape:', first_batch_shape)
display(model_window_summary_df)

## Flattened-Window XGBoost

Το XGBoost ξεκινά από το ίδιο sequence tensor `[n_windows, 24, 41]` και το κάνει flatten σε `[n_windows, 984]`. Η επιλογή configuration γίνεται μόνο στο validation split, με ranking `MAE` ascending, `RMSE` ascending και `R2` descending.

In [ ]:
# ============================================================
# NB19 | Flattened-window XGBoost helpers, grid και shape check
# ============================================================

def flatten_sequence_windows(X: np.ndarray, expected_features: int) -> np.ndarray:
    if X.ndim != 3:
        raise ValueError(f'Το XGBoost flattening αναμένει [n_windows, lookback, features], πήρε {X.shape}.')
    if X.shape[1] != LOOKBACK_STEPS or X.shape[2] != expected_features:
        raise ValueError(
            f'Μη αναμενόμενο sequence shape για flattening: {X.shape}; '
            f'αναμενόταν (*, {LOOKBACK_STEPS}, {expected_features}).'
        )
    return X.reshape(X.shape[0], X.shape[1] * X.shape[2]).astype(np.float32)


dummy_sequence_for_xgboost = np.zeros((8, LOOKBACK_STEPS, EXPECTED_FEATURE_COUNT), dtype=np.float32)
dummy_flattened_for_xgboost = flatten_sequence_windows(
    dummy_sequence_for_xgboost,
    expected_features=EXPECTED_FEATURE_COUNT,
)
xgboost_dummy_shape = dummy_flattened_for_xgboost.shape
if xgboost_dummy_shape != (8, EXPECTED_FLATTENED_FEATURE_COUNT):
    raise ValueError(f'Απέτυχε το dummy XGBoost flatten shape check: {xgboost_dummy_shape}')
del dummy_sequence_for_xgboost, dummy_flattened_for_xgboost

xgboost_validation_metrics_df = pd.DataFrame()
xgboost_selected_test_metrics_df = pd.DataFrame()
selected_xgboost_model: Any | None = None
selected_xgboost_metadata: dict[str, Any] = {}
selected_xgboost_config_id: int | None = None
xgboost_test_evaluations = 0

if RUN_SMOKE_TRAINING:
    import xgboost as xgb

    X_train_xgb = flatten_sequence_windows(X_train_model, expected_features=len(numeric_feature_cols))
    X_val_xgb = flatten_sequence_windows(X_val_model, expected_features=len(numeric_feature_cols))
    X_test_xgb = flatten_sequence_windows(X_test_model, expected_features=len(numeric_feature_cols))

    candidate_xgboost_models: dict[int, Any] = {}
    metric_rows: list[dict[str, Any]] = []
    grid_values = itertools.product(
        XGBOOST_GRID['n_estimators'],
        XGBOOST_GRID['max_depth'],
        XGBOOST_GRID['learning_rate'],
        XGBOOST_GRID['subsample'],
        XGBOOST_GRID['colsample_bytree'],
    )

    for config_id, (n_estimators, max_depth, learning_rate, subsample, colsample_bytree) in enumerate(grid_values, start=1):
        run_start = time.perf_counter()
        model = xgb.XGBRegressor(
            objective='reg:squarederror',
            random_state=SEED,
            n_jobs=-1,
            tree_method='hist',
            eval_metric='rmse',
            n_estimators=int(n_estimators),
            max_depth=int(max_depth),
            learning_rate=float(learning_rate),
            subsample=float(subsample),
            colsample_bytree=float(colsample_bytree),
        )
        model.fit(X_train_xgb, y_train_model)

        val_pred = model.predict(X_val_xgb)
        val_metrics = compute_metrics(y_val_model, val_pred)
        elapsed_seconds = time.perf_counter() - run_start

        candidate_xgboost_models[config_id] = model
        metric_rows.append(
            {
                'run_mode': 'smoke' if SMOKE_MODE else 'subset',
                'evidence_status': 'έλεγχος κώδικα μόνο, όχι τεκμήριο manuscript' if SMOKE_MODE else 'τεκμήριο για το subset μόνο, όχι αντικατάσταση benchmark',
                'model': 'Flattened-window XGBoost',
                'model_id': 'flattened_window_xgboost',
                'config_id': config_id,
                'selected_parks': ';'.join(SELECTED_PARKS),
                'n_parks': len(SELECTED_PARKS),
                'seed': SEED,
                'lookback_steps': LOOKBACK_STEPS,
                'n_numeric_features': len(numeric_feature_cols),
                'flattened_features': X_train_xgb.shape[1],
                'n_estimators': int(n_estimators),
                'max_depth': int(max_depth),
                'learning_rate': float(learning_rate),
                'subsample': float(subsample),
                'colsample_bytree': float(colsample_bytree),
                'train_windows_used': len(X_train_xgb),
                'val_windows_used': len(X_val_xgb),
                'test_windows_available': len(X_test_xgb),
                'MAE': val_metrics['MAE'],
                'RMSE': val_metrics['RMSE'],
                'R2': val_metrics['R2'],
                'selection_split': 'validation',
                'selection_policy': 'validation ranking: MAE primary, then RMSE, then R2',
                'elapsed_seconds': round(elapsed_seconds, 3),
            }
        )

    xgboost_validation_metrics_df = rank_validation_metrics(pd.DataFrame(metric_rows))
    selected_xgboost_metadata = xgboost_validation_metrics_df.iloc[0].to_dict()
    selected_xgboost_config_id = int(selected_xgboost_metadata['config_id'])
    selected_xgboost_model = candidate_xgboost_models[selected_xgboost_config_id]
    display(xgboost_validation_metrics_df)
else:
    print('RUN_SMOKE_TRAINING είναι False· δεν εκτελέστηκε XGBoost validation grid.')

if RUN_TEST_EVALUATION:
    if selected_xgboost_model is None:
        raise RuntimeError('RUN_TEST_EVALUATION=True απαιτεί validation-selected XGBoost configuration στη μνήμη.')

    test_pred = selected_xgboost_model.predict(X_test_xgb)
    test_metrics = compute_metrics(y_test_model, test_pred)
    xgboost_test_evaluations = 1
    xgboost_selected_test_metrics_df = pd.DataFrame(
        [
            {
                'run_mode': 'smoke' if SMOKE_MODE else 'subset',
                'evidence_status': 'έλεγχος κώδικα μόνο, όχι τεκμήριο manuscript' if SMOKE_MODE else 'τεκμήριο για το subset μόνο, όχι αντικατάσταση benchmark',
                'model': 'Flattened-window XGBoost',
                'model_id': 'flattened_window_xgboost',
                'config_id': selected_xgboost_config_id,
                'selected_parks': ';'.join(SELECTED_PARKS),
                'n_parks': len(SELECTED_PARKS),
                'seed': SEED,
                'lookback_steps': LOOKBACK_STEPS,
                'n_numeric_features': len(numeric_feature_cols),
                'flattened_features': EXPECTED_FLATTENED_FEATURE_COUNT,
                'n_estimators': selected_xgboost_metadata.get('n_estimators'),
                'max_depth': selected_xgboost_metadata.get('max_depth'),
                'learning_rate': selected_xgboost_metadata.get('learning_rate'),
                'subsample': selected_xgboost_metadata.get('subsample'),
                'colsample_bytree': selected_xgboost_metadata.get('colsample_bytree'),
                'train_windows_used': len(X_train_model),
                'val_windows_used': len(X_val_model),
                'test_windows_used': len(X_test_model),
                'MAE': test_metrics['MAE'],
                'RMSE': test_metrics['RMSE'],
                'R2': test_metrics['R2'],
                'test_evaluations': xgboost_test_evaluations,
                'test_policy': 'το validation-selected configuration αξιολογείται μία φορά στο test subset',
            }
        ]
    )
    display(xgboost_selected_test_metrics_df)
else:
    print('RUN_TEST_EVALUATION είναι False· το XGBoost test split δεν αξιολογήθηκε.')

print('Dummy XGBoost flatten shape:', xgboost_dummy_shape)

## PatchTST-Lite Μοντελοποίηση

Το PatchTST-Lite υλοποιείται μόνο με PyTorch. Η είσοδος `[batch, 24, 41]` γίνεται patchify στο time dimension, περνά από linear patch projection, positional embedding, `nn.TransformerEncoder` και regression head.

In [ ]:
# ============================================================
# NB19 | PatchTST-Lite model, metrics και training helpers
# ============================================================

class PatchTSTLiteRegressor(nn.Module):
    def __init__(
        self,
        input_dim: int,
        lookback_steps: int,
        patch_len: int,
        patch_stride: int,
        d_model: int,
        n_heads: int,
        n_layers: int,
        dropout: float,
    ) -> None:
        super().__init__()
        if lookback_steps < patch_len:
            raise ValueError('Το patch_len δεν μπορεί να είναι μεγαλύτερο από το lookback.')
        if d_model % n_heads != 0:
            raise ValueError('Το d_model πρέπει να διαιρείται ακριβώς από το n_heads.')

        self.input_dim = input_dim
        self.lookback_steps = lookback_steps
        self.patch_len = patch_len
        self.patch_stride = patch_stride
        self.num_patches = 1 + (lookback_steps - patch_len) // patch_stride

        patch_width = patch_len * input_dim
        self.patch_projection = nn.Linear(patch_width, d_model)
        self.positional_embedding = nn.Parameter(torch.zeros(1, self.num_patches, d_model))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=4 * d_model,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(d_model, 1),
        )
        self.reset_parameters()

    def reset_parameters(self) -> None:
        nn.init.normal_(self.positional_embedding, mean=0.0, std=0.02)
        nn.init.xavier_uniform_(self.patch_projection.weight)
        nn.init.zeros_(self.patch_projection.bias)
        final_linear = self.head[-1]
        if isinstance(final_linear, nn.Linear):
            nn.init.xavier_uniform_(final_linear.weight)
            nn.init.zeros_(final_linear.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.ndim != 3:
            raise ValueError(f'Το PatchTST-Lite αναμένει [batch, lookback, features], πήρε {tuple(x.shape)}.')
        if x.shape[1] != self.lookback_steps or x.shape[2] != self.input_dim:
            raise ValueError(
                f'Μη αναμενόμενο PatchTST-Lite input shape={tuple(x.shape)}; '
                f'αναμενόταν (*, {self.lookback_steps}, {self.input_dim}).'
            )

        patches = x.unfold(dimension=1, size=self.patch_len, step=self.patch_stride)
        patches = patches.permute(0, 1, 3, 2).contiguous()
        patches = patches.view(x.shape[0], self.num_patches, self.patch_len * self.input_dim)
        tokens = self.patch_projection(patches) + self.positional_embedding
        encoded = self.encoder(tokens)
        pooled = self.norm(encoded).mean(dim=1)
        return self.head(pooled)


def run_epoch(model: nn.Module, loader: DataLoader, criterion: nn.Module, optimizer: Any | None = None) -> float:
    is_train = optimizer is not None
    model.train(mode=is_train)
    total_loss = 0.0
    total_count = 0

    for xb, yb in loader:
        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)

        if is_train:
            optimizer.zero_grad()

        pred = model(xb)
        loss = criterion(pred, yb)

        if is_train:
            loss.backward()
            optimizer.step()

        batch_size_current = xb.shape[0]
        total_loss += float(loss.detach().cpu().item()) * batch_size_current
        total_count += batch_size_current

    if total_count == 0:
        raise ValueError('Δεν μπορεί να τρέξει epoch σε empty loader.')
    return total_loss / total_count


@torch.no_grad()
def predict_loader(model: nn.Module, loader: DataLoader) -> np.ndarray:
    model.eval()
    preds: list[np.ndarray] = []
    for xb, _ in loader:
        xb = xb.to(DEVICE)
        pred = model(xb).detach().cpu().numpy().reshape(-1)
        preds.append(pred)
    if not preds:
        raise ValueError('Δεν μπορεί να γίνει prediction από empty loader.')
    return np.concatenate(preds).astype(float)


def validation_ranking_improved(candidate_metrics: dict[str, float], best_metrics: dict[str, float] | None) -> bool:
    if best_metrics is None:
        return True
    if candidate_metrics['MAE'] < best_metrics['MAE'] - 1e-8:
        return True
    if np.isclose(candidate_metrics['MAE'], best_metrics['MAE'], atol=1e-8, rtol=0.0):
        if candidate_metrics['RMSE'] < best_metrics['RMSE'] - 1e-8:
            return True
        if np.isclose(candidate_metrics['RMSE'], best_metrics['RMSE'], atol=1e-8, rtol=0.0):
            return candidate_metrics['R2'] > best_metrics['R2'] + 1e-8
    return False


def train_patchtst_with_early_stopping(
    train_loader: DataLoader,
    val_loader: DataLoader,
    y_val: np.ndarray,
    input_dim: int,
    epochs: int,
    patience: int,
) -> tuple[nn.Module, pd.DataFrame, int, float, dict[str, float]]:
    set_reproducibility(SEED)
    model = PatchTSTLiteRegressor(
        input_dim=input_dim,
        lookback_steps=LOOKBACK_STEPS,
        patch_len=PATCH_LEN,
        patch_stride=PATCH_STRIDE,
        d_model=D_MODEL,
        n_heads=N_HEADS,
        n_layers=N_LAYERS,
        dropout=DROPOUT,
    ).to(DEVICE)
    criterion = nn.MSELoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

    best_state = None
    best_epoch = 0
    best_val_loss_at_selected = float('inf')
    best_validation_metrics: dict[str, float] | None = None
    epochs_without_improvement = 0
    history_rows: list[dict[str, Any]] = []

    for epoch in range(1, epochs + 1):
        train_loss = run_epoch(model, train_loader, criterion, optimizer=optimizer)
        val_loss = run_epoch(model, val_loader, criterion, optimizer=None)
        val_pred = predict_loader(model, val_loader)
        val_metrics = compute_metrics(y_val, val_pred)
        improved = validation_ranking_improved(val_metrics, best_validation_metrics)

        if improved:
            best_val_loss_at_selected = val_loss
            best_validation_metrics = val_metrics
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        history_rows.append(
            {
                'run_mode': 'smoke' if SMOKE_MODE else 'subset',
                'evidence_status': 'έλεγχος κώδικα μόνο, όχι τεκμήριο manuscript' if SMOKE_MODE else 'τεκμήριο για το subset μόνο, όχι αντικατάσταση benchmark',
                'model': 'PatchTST-Lite Transformer',
                'model_id': PATCHTST_MODEL_ID,
                'epoch': epoch,
                'train_loss_mse': train_loss,
                'val_loss_mse': val_loss,
                'val_MAE': val_metrics['MAE'],
                'val_RMSE': val_metrics['RMSE'],
                'val_R2': val_metrics['R2'],
                'selection_metric_primary': 'validation_MAE',
                'is_best_epoch': improved,
            }
        )

        if epochs_without_improvement >= patience:
            break

    if best_state is None or best_validation_metrics is None:
        raise RuntimeError('No validation-selected PatchTST-Lite state was captured.')

    model.load_state_dict(best_state)
    return model, pd.DataFrame(history_rows), best_epoch, float(best_val_loss_at_selected), dict(best_validation_metrics)


dummy_patchtst_model = PatchTSTLiteRegressor(
    input_dim=EXPECTED_FEATURE_COUNT,
    lookback_steps=LOOKBACK_STEPS,
    patch_len=PATCH_LEN,
    patch_stride=PATCH_STRIDE,
    d_model=D_MODEL,
    n_heads=N_HEADS,
    n_layers=N_LAYERS,
    dropout=DROPOUT,
).to(DEVICE)
dummy_patchtst_model.eval()
dummy_patchtst_input = torch.zeros((8, LOOKBACK_STEPS, EXPECTED_FEATURE_COUNT), dtype=torch.float32, device=DEVICE)
with torch.no_grad():
    dummy_patchtst_output = dummy_patchtst_model(dummy_patchtst_input)
patchtst_dummy_input_shape = tuple(dummy_patchtst_input.shape)
patchtst_dummy_output_shape = tuple(dummy_patchtst_output.shape)
if patchtst_dummy_output_shape != (8, 1):
    raise ValueError(f'Απέτυχε το dummy PatchTST shape check: {patchtst_dummy_output_shape}')
del dummy_patchtst_model, dummy_patchtst_input, dummy_patchtst_output

print('Dummy PatchTST input shape:', patchtst_dummy_input_shape)
print('Dummy PatchTST output shape:', patchtst_dummy_output_shape)

## Smoke-Safe Εκπαίδευση PatchTST-Lite

Το `RUN_SMOKE_TRAINING` είναι προεπιλεγμένα `False`, άρα η εκτέλεση του notebook δεν εκπαιδεύει μοντέλο. Αν ενεργοποιηθεί αργότερα, το `SMOKE_MODE` κρατά capped windows και λίγα epochs για έλεγχο κώδικα. Για full local subset run, τα flags αλλάζουν χειροκίνητα.

In [ ]:
# ============================================================
# NB19 | Validation-only επιλογή PatchTST-Lite
# ============================================================

patchtst_selected_model: nn.Module | None = None
patchtst_selected_model_metadata: dict[str, Any] = {}
patchtst_training_history_df = pd.DataFrame()
patchtst_validation_metrics_df = pd.DataFrame()
patchtst_best_epoch: int | None = None
patchtst_best_val_loss_mse: float | None = None
patchtst_best_validation_metrics: dict[str, float] = {}

if RUN_SMOKE_TRAINING:
    effective_epochs = SMOKE_EPOCHS if SMOKE_MODE else EPOCHS_REQUESTED
    effective_patience = SMOKE_PATIENCE if SMOKE_MODE else EARLY_STOPPING_PATIENCE
    run_start = time.perf_counter()

    patchtst_selected_model, patchtst_training_history_df, patchtst_best_epoch, patchtst_best_val_loss_mse, patchtst_best_validation_metrics = train_patchtst_with_early_stopping(
        train_loader=train_loader,
        val_loader=val_loader,
        y_val=y_val_model,
        input_dim=len(numeric_feature_cols),
        epochs=effective_epochs,
        patience=effective_patience,
    )

    val_metrics = patchtst_best_validation_metrics
    elapsed_seconds = time.perf_counter() - run_start

    patchtst_validation_metrics_df = rank_validation_metrics(
        pd.DataFrame(
            [
                {
                    'run_mode': 'smoke' if SMOKE_MODE else 'subset',
                    'evidence_status': 'έλεγχος κώδικα μόνο, όχι τεκμήριο manuscript' if SMOKE_MODE else 'τεκμήριο για το subset μόνο, όχι αντικατάσταση benchmark',
                    'model': 'PatchTST-Lite Transformer',
                    'model_id': PATCHTST_MODEL_ID,
                    'selected_parks': ';'.join(SELECTED_PARKS),
                    'n_parks': len(SELECTED_PARKS),
                    'seed': SEED,
                    'device': str(DEVICE),
                    'lookback_steps': LOOKBACK_STEPS,
                    'patch_len': PATCH_LEN,
                    'patch_stride': PATCH_STRIDE,
                    'd_model': D_MODEL,
                    'n_heads': N_HEADS,
                    'n_layers': N_LAYERS,
                    'dropout': DROPOUT,
                    'learning_rate': LEARNING_RATE,
                    'weight_decay': WEIGHT_DECAY,
                    'batch_size': BATCH_SIZE,
                    'epochs_requested': effective_epochs,
                    'best_epoch': patchtst_best_epoch,
                    'best_val_loss_mse': patchtst_best_val_loss_mse,
                    'n_numeric_features': len(numeric_feature_cols),
                    'train_windows_used': len(X_train_model),
                    'val_windows_used': len(X_val_model),
                    'test_windows_available': len(X_test_model),
                    'MAE': val_metrics['MAE'],
                    'RMSE': val_metrics['RMSE'],
                    'R2': val_metrics['R2'],
                    'selection_split': 'validation',
                    'selection_metric_primary': 'validation_MAE',
                    'selection_policy': 'validation ranking: MAE primary, then RMSE, then R2',
                    'elapsed_seconds': round(elapsed_seconds, 3),
                }
            ]
        )
    )
    patchtst_selected_model_metadata = patchtst_validation_metrics_df.iloc[0].to_dict()

    display(patchtst_training_history_df)
    display(patchtst_validation_metrics_df)
else:
    print('RUN_SMOKE_TRAINING είναι False· δεν εκτελέστηκε PatchTST-Lite training.')

## Πύλη Εφάπαξ Test Evaluation

Το `RUN_TEST_EVALUATION` είναι προεπιλεγμένα `False`. Αν ενεργοποιηθεί αφού υπάρχει validation-selected PatchTST-Lite state, το επιλεγμένο state αξιολογείται μία φορά στο test subset. Δεν γίνεται test-driven model selection.

In [ ]:
# ============================================================
# NB19 | Πύλη τελικής test-only αξιολόγησης PatchTST-Lite
# ============================================================

patchtst_selected_test_metrics_df = pd.DataFrame()
patchtst_test_evaluations = 0

if RUN_TEST_EVALUATION:
    if patchtst_selected_model is None:
        raise RuntimeError('RUN_TEST_EVALUATION=True απαιτεί validation-selected PatchTST-Lite model στη μνήμη.')

    test_pred = predict_loader(patchtst_selected_model, test_loader)
    test_metrics = compute_metrics(y_test_model, test_pred)
    patchtst_test_evaluations = 1

    patchtst_selected_test_metrics_df = pd.DataFrame(
        [
            {
                'run_mode': 'smoke' if SMOKE_MODE else 'subset',
                'evidence_status': 'έλεγχος κώδικα μόνο, όχι τεκμήριο manuscript' if SMOKE_MODE else 'τεκμήριο για το subset μόνο, όχι αντικατάσταση benchmark',
                'model': 'PatchTST-Lite Transformer',
                'model_id': PATCHTST_MODEL_ID,
                'selected_parks': ';'.join(SELECTED_PARKS),
                'n_parks': len(SELECTED_PARKS),
                'seed': SEED,
                'device': str(DEVICE),
                'lookback_steps': LOOKBACK_STEPS,
                'patch_len': PATCH_LEN,
                'patch_stride': PATCH_STRIDE,
                'd_model': D_MODEL,
                'n_heads': N_HEADS,
                'n_layers': N_LAYERS,
                'dropout': DROPOUT,
                'learning_rate': LEARNING_RATE,
                'weight_decay': WEIGHT_DECAY,
                'batch_size': BATCH_SIZE,
                'epochs_requested': SMOKE_EPOCHS if SMOKE_MODE else EPOCHS_REQUESTED,
                'best_epoch': patchtst_best_epoch,
                'best_val_loss_mse': patchtst_best_val_loss_mse,
                'n_numeric_features': len(numeric_feature_cols),
                'train_windows_used': len(X_train_model),
                'val_windows_used': len(X_val_model),
                'test_windows_used': len(X_test_model),
                'MAE': test_metrics['MAE'],
                'RMSE': test_metrics['RMSE'],
                'R2': test_metrics['R2'],
                'test_evaluations': patchtst_test_evaluations,
                'test_policy': 'το validation-selected state αξιολογείται μία φορά στο test subset',
            }
        ]
    )
    display(patchtst_selected_test_metrics_df)
else:
    print('RUN_TEST_EVALUATION είναι False· το PatchTST-Lite test split δεν αξιολογήθηκε.')

## Προαιρετικά Local-Only Exports

Τα exports είναι απενεργοποιημένα από προεπιλογή. Αν ενεργοποιηθούν αργότερα, γράφονται μόνο τα δηλωμένα CSV paths κάτω από `data/processed/diagnostics/nn_sequence_subset_patchtst/`. Δεν γράφονται checkpoints ή binary model files.

In [ ]:
# ============================================================
# NB19 | Προαιρετικά local-only CSV exports
# ============================================================

exported_csv_paths: list[Path] = []

run_manifest_df = pd.DataFrame(
    [
        {'field': 'notebook', 'value': 'notebooks/19_patchtst_sequence_subset.ipynb'},
        {'field': 'evidence_status', 'value': 'έλεγχος κώδικα μόνο, όχι τεκμήριο manuscript' if SMOKE_MODE else 'τεκμήριο για το subset μόνο, όχι αντικατάσταση benchmark'},
        {'field': 'selected_parks', 'value': ';'.join(SELECTED_PARKS)},
        {'field': 'target_column', 'value': TARGET_COLUMN},
        {'field': 'excluded_columns', 'value': ', '.join(sorted(EXCLUDED_COLUMNS))},
        {'field': 'feature_contract', 'value': 'μόνο numeric learned train columns, χωρίς turbine encoding και χωρίς validation/test statistics για feature selection'},
        {'field': 'scaling_policy', 'value': 'StandardScaler fit μόνο σε train subset feature rows, target unscaled'},
        {'field': 'sequence_policy', 'value': 'X rows [i..i+23] προβλέπουν target row i+24 για LOOKBACK_STEPS=24'},
        {'field': 'xgboost_policy', 'value': 'flatten [n_windows, 24, 41] σε [n_windows, 984] και validation-only grid selection'},
        {'field': 'patchtst_policy', 'value': 'pure PyTorch PatchTST-Lite με patchify, transformer encoder και regression head'},
        {'field': 'selection_policy', 'value': 'validation ranking: MAE primary, then RMSE, then R2'},
        {'field': 'test_policy', 'value': 'το validation-selected configuration/state αξιολογείται μία φορά στο test μόνο αν RUN_TEST_EVALUATION=True'},
        {'field': 'baseline_metrics_policy', 'value': 'το data/processed/baseline_metrics.csv δεν τροποποιείται'},
        {'field': 'model_artifact_policy', 'value': 'δεν γράφονται checkpoints ή model binaries'},
        {'field': 'export_results', 'value': EXPORT_RESULTS},
    ]
)

if EXPORT_RESULTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    run_manifest_df.to_csv(RUN_MANIFEST_PATH, index=False)
    window_audit_df.to_csv(WINDOW_AUDIT_PATH, index=False)
    xgboost_validation_metrics_df.to_csv(XGBOOST_VALIDATION_METRICS_PATH, index=False)
    xgboost_selected_test_metrics_df.to_csv(XGBOOST_SELECTED_TEST_METRICS_PATH, index=False)
    patchtst_training_history_df.to_csv(TRAINING_HISTORY_PATH, index=False)
    patchtst_validation_metrics_df.to_csv(VALIDATION_METRICS_PATH, index=False)
    patchtst_selected_test_metrics_df.to_csv(SELECTED_TEST_METRICS_PATH, index=False)

    exported_csv_paths = [
        RUN_MANIFEST_PATH,
        WINDOW_AUDIT_PATH,
        XGBOOST_VALIDATION_METRICS_PATH,
        XGBOOST_SELECTED_TEST_METRICS_PATH,
        TRAINING_HISTORY_PATH,
        VALIDATION_METRICS_PATH,
        SELECTED_TEST_METRICS_PATH,
    ]
    print('Γράφτηκαν local-only NB19 exports:')
    for path in exported_csv_paths:
        print('-', path)
else:
    print('EXPORT_RESULTS είναι False· δεν γράφτηκαν παραγόμενα CSV αρχεία.')

## Τελική Σύνοψη Ελέγχων

Το τελευταίο cell συγκεντρώνει τους ελέγχους scaffold/full-run mode: canonical inputs, feature count, shape contracts, run flags, export policy και προστασία του `baseline_metrics.csv`.

In [ ]:
# ============================================================
# NB19 | Τελική σύνοψη ελέγχων
# ============================================================

expected_output_dir = DATA_PROCESSED / 'diagnostics' / 'nn_sequence_subset_patchtst'
expected_output_dir_resolved = expected_output_dir.resolve()

exported_paths_under_expected_dir = all(
    path.suffix == '.csv' and expected_output_dir_resolved in path.resolve().parents
    for path in exported_csv_paths
)

export_results_policy_ok = (
    (EXPORT_RESULTS is False)
    or (
        OUTPUT_DIR.resolve() == expected_output_dir_resolved
        and len(exported_csv_paths) == 7
        and exported_paths_under_expected_dir
    )
)

baseline_metrics_current_state = get_file_state(BASELINE_METRICS_PATH)
baseline_metrics_not_modified = baseline_metrics_current_state == BASELINE_METRICS_INITIAL_STATE

window_counts_positive_when_data_exists = all(
    (len(split_frames[split_name]) == 0) or (len(X_seq) > 0)
    for split_name, X_seq in [
        ('train', X_train_seq),
        ('validation', X_val_seq),
        ('test', X_test_seq),
    ]
)

first_batch_shape_ok = (
    len(first_batch_shape) == 3
    and first_batch_shape[1] == LOOKBACK_STEPS
    and first_batch_shape[2] == len(numeric_feature_cols)
    and first_batch_shape[2] == EXPECTED_FEATURE_COUNT
)

scaffold_safe_mode_ok = (
    SMOKE_MODE is True
    and RUN_SMOKE_TRAINING is False
    and RUN_TEST_EVALUATION is False
    and EXPORT_RESULTS is False
)
full_results_mode_ok = (
    SMOKE_MODE is False
    and RUN_SMOKE_TRAINING is True
    and RUN_TEST_EVALUATION is True
    and EXPORT_RESULTS is True
)
run_flags_mode_ok = scaffold_safe_mode_ok or full_results_mode_ok
expected_test_evaluations = 1 if RUN_TEST_EVALUATION else 0
xgboost_test_evaluation_count_ok = xgboost_test_evaluations == expected_test_evaluations
patchtst_test_evaluation_count_ok = patchtst_test_evaluations == expected_test_evaluations

dummy_flatten_shape_ok = xgboost_dummy_shape == (8, EXPECTED_FLATTENED_FEATURE_COUNT)
dummy_patchtst_shape_ok = patchtst_dummy_input_shape == (8, LOOKBACK_STEPS, EXPECTED_FEATURE_COUNT) and patchtst_dummy_output_shape == (8, 1)
no_target_leakage = TARGET_COLUMN not in numeric_feature_cols and not (set(numeric_feature_cols) & EXCLUDED_COLUMNS)

verification_rows = [
    {'check': 'selected_parks', 'status': SELECTED_PARKS == EXPECTED_SELECTED_PARKS, 'detail': ';'.join(SELECTED_PARKS)},
    {'check': 'lookback_steps', 'status': LOOKBACK_STEPS == 24, 'detail': f'LOOKBACK_STEPS={LOOKBACK_STEPS}'},
    {'check': 'split_paths_exist', 'status': all(path.exists() for path in [TRAIN_PATH, VAL_PATH, TEST_PATH]), 'detail': 'train/validation/test canonical paths'},
    {'check': 'feature_count_train_inferred', 'status': FEATURE_SELECTION_SOURCE == 'train' and len(numeric_feature_cols) == EXPECTED_FEATURE_COUNT, 'detail': f'n_features={len(numeric_feature_cols)}'},
    {'check': 'no_target_leakage', 'status': no_target_leakage, 'detail': TARGET_COLUMN},
    {'check': 'scaler_fit_train_only', 'status': SCALER_FIT_SOURCE == 'train' and getattr(scaler, 'n_features_in_', None) == len(numeric_feature_cols), 'detail': 'StandardScaler fit στο train subset'},
    {'check': 'window_counts_positive_when_data_exists', 'status': window_counts_positive_when_data_exists, 'detail': f'train={len(X_train_seq)}, validation={len(X_val_seq)}, test={len(X_test_seq)}'},
    {'check': 'first_batch_shape', 'status': first_batch_shape_ok, 'detail': str(first_batch_shape)},
    {'check': 'dummy_xgboost_flatten_shape', 'status': dummy_flatten_shape_ok, 'detail': str(xgboost_dummy_shape)},
    {'check': 'dummy_patchtst_shape', 'status': dummy_patchtst_shape_ok, 'detail': f'{patchtst_dummy_input_shape} -> {patchtst_dummy_output_shape}'},
    {'check': 'run_flags_mode_ok', 'status': run_flags_mode_ok, 'detail': f'scaffold_safe_mode_ok={scaffold_safe_mode_ok}, full_results_mode_ok={full_results_mode_ok}, SMOKE_MODE={SMOKE_MODE}, RUN_SMOKE_TRAINING={RUN_SMOKE_TRAINING}, RUN_TEST_EVALUATION={RUN_TEST_EVALUATION}, EXPORT_RESULTS={EXPORT_RESULTS}'},
    {'check': 'baseline_metrics_not_modified', 'status': baseline_metrics_not_modified, 'detail': str(BASELINE_METRICS_PATH)},
    {'check': 'export_results_policy', 'status': export_results_policy_ok, 'detail': f'OUTPUT_DIR={OUTPUT_DIR}'},
    {'check': 'validation_ranking_policy', 'status': VALIDATION_RANKING_COLUMNS == ['MAE', 'RMSE', 'R2'] and VALIDATION_RANKING_ASCENDING == [True, True, False], 'detail': 'MAE asc, RMSE asc, R2 desc'},
    {'check': 'xgboost_test_evaluation_count', 'status': xgboost_test_evaluation_count_ok, 'detail': f'expected={expected_test_evaluations}, observed={xgboost_test_evaluations}'},
    {'check': 'patchtst_test_evaluation_count', 'status': patchtst_test_evaluation_count_ok, 'detail': f'expected={expected_test_evaluations}, observed={patchtst_test_evaluations}'},
    {'check': 'no_checkpoint_paths_defined', 'status': True, 'detail': 'Δεν ορίζονται model save/checkpoint path constants.'},
]

verification_df = pd.DataFrame(verification_rows)
display(verification_df)

if not verification_df['status'].astype(bool).all():
    raise ValueError('Απέτυχε ένας ή περισσότεροι τελικοί έλεγχοι του NB19.')

print('Ο τελικός έλεγχος του NB19 ολοκληρώθηκε.')
print('Παραγόμενα CSV αρχεία που γράφτηκαν:', len(exported_csv_paths))
print('Checkpoint/model binary outputs που γράφτηκαν: 0')